# DỰ ĐOÁN BURNOUT LEVEL BẰNG MACHINE LEARNING

**Target:** `burnout_level`

**Bài toán:** Hồi quy Logistic

**Các thuật toán sử dụng:**
- Logistic Regression
- Random Forest
- Catboost

1. Đọc và kiểm tra dữ liệu

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

train_log_df = pd.read_csv('../data/data_normalized/burnout_train_standardized.csv')
test_log_df = pd.read_csv('../data/data_normalized/burnout_test_standardized.csv')

train_df = pd.read_csv('../data/data_normalized/burnout_train.csv')
test_df = pd.read_csv('../data/data_normalized/burnout_test.csv')

print("Train Standardized:", train_log_df.shape)
print("Test Standardized:", test_log_df.shape)

print("Train:", train_df.shape)
print("Test:", test_df.shape)

Train Standardized: (80000, 64)
Test Standardized: (20000, 64)
Train: (80000, 64)
Test: (20000, 64)


In [6]:
SELECTED_FEATURES = [
    'work_hours_per_week',
    'sleep_hours_per_night',
    'work_life_balance_ratio',
    'deadline_pressure_score',
    'stress_score',
    'phq9_score',
    'gad7_score',
    'years_experience',
    'uses_therapy',
    'seeks_mental_health_support',
    'job_change_intention'
]

def select_features(df):
    selected_cols = [
        col for col in df.columns
        if any(
            col == feature
            or col == f'numeric__{feature}'
            or col.startswith(f'categorical__{feature}_')
            for feature in SELECTED_FEATURES
        )
    ]

    return df[selected_cols].copy()

X_train_log = select_features(train_log_df)
X_test_log = select_features(test_log_df)

X_train = select_features(train_df)
X_test = select_features(test_df)

y_train = train_df['burnout_level']
y_test = test_df['burnout_level']

assert X_train.columns.equals(X_test.columns)
assert X_train_log.columns.equals(X_test_log.columns)

print("Số features Logistic:", X_train_log.shape[1])
print("Số features RF/CatBoost:", X_train.shape[1])

print("\nDanh sách feature được chọn:")
print(X_train.columns.tolist())

Số features Logistic: 11
Số features RF/CatBoost: 11

Danh sách feature được chọn:
['numeric__years_experience', 'numeric__work_hours_per_week', 'numeric__sleep_hours_per_night', 'numeric__work_life_balance_ratio', 'numeric__uses_therapy', 'numeric__deadline_pressure_score', 'numeric__stress_score', 'numeric__phq9_score', 'numeric__gad7_score', 'numeric__seeks_mental_health_support', 'numeric__job_change_intention']


 2. Xây dựng mô hình 
 
 2.1. Xây dựng mô hình Logistic Regression

In [7]:
log_model = LogisticRegression(
    max_iter=1000,
    class_weight={
            'Low': 1,
            'Moderate': 1,
            'High': 1.4,
            'Severe': 1
        },    
    random_state=42
)
log_model.fit(X_train_log, y_train)
y_pred_log = log_model.predict(X_test_log)
accuracy_log = accuracy_score(y_test, y_pred_log)

print("Accuracy Logistic Regression:", round(accuracy_log, 4))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_log))

c:\Users\nnhat\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Accuracy Logistic Regression: 0.7138

Classification Report:
              precision    recall  f1-score   support

        High       0.50      0.58      0.54      3873
         Low       0.83      0.82      0.82      5161
    Moderate       0.63      0.59      0.61      5251
      Severe       0.86      0.82      0.84      5715

    accuracy                           0.71     20000
   macro avg       0.70      0.70      0.70     20000
weighted avg       0.72      0.71      0.72     20000



2.2. RandomForestClassifier

In [8]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
accuracy_rf = accuracy_score(y_test, y_pred_rf)

print("Accuracy Random Forest:", round(accuracy_rf, 4))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

Accuracy Random Forest: 0.7109

Classification Report:
              precision    recall  f1-score   support

        High       0.52      0.43      0.47      3873
         Low       0.84      0.80      0.82      5161
    Moderate       0.60      0.68      0.64      5251
      Severe       0.83      0.85      0.84      5715

    accuracy                           0.71     20000
   macro avg       0.69      0.69      0.69     20000
weighted avg       0.71      0.71      0.71     20000



2.3. CatBoost

In [9]:
from catboost import CatBoostClassifier

# 1. Khởi tạo mô hình
cat_model = CatBoostClassifier(
    iterations=100,
    depth=6,
    learning_rate=0.1,
    random_seed=42,
    verbose=0
)
cat_model.fit(X_train, y_train)
y_pred_cat = cat_model.predict(X_test).ravel()
accuracy_cat = accuracy_score(y_test, y_pred_cat)

print("Accuracy CatBoost:", round(accuracy_cat, 4))
print("Classification Report:")
print(classification_report(y_test, y_pred_cat))

Accuracy CatBoost: 0.7176
Classification Report:
              precision    recall  f1-score   support

        High       0.52      0.48      0.50      3873
         Low       0.84      0.81      0.82      5161
    Moderate       0.61      0.66      0.63      5251
      Severe       0.84      0.85      0.84      5715

    accuracy                           0.72     20000
   macro avg       0.70      0.70      0.70     20000
weighted avg       0.72      0.72      0.72     20000



3. So sánh các mô hình

3.1. So sánh

In [10]:
results = pd.DataFrame({
    'Model': [
        'Logistic Regression',
        'Random Forest',
        'CatBoost'
    ],
    'Accuracy': [
        accuracy_score(y_test, y_pred_log),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_cat)
    ],
    'Precision': [
        precision_score(y_test, y_pred_log, average='weighted'),
        precision_score(y_test, y_pred_rf, average='weighted'),
        precision_score(y_test, y_pred_cat, average='weighted')
    ],
    'Recall': [
        recall_score(y_test, y_pred_log, average='weighted'),
        recall_score(y_test, y_pred_rf, average='weighted'),
        recall_score(y_test, y_pred_cat, average='weighted')
    ],
    'F1-Score': [
        f1_score(y_test, y_pred_log, average='weighted'),
        f1_score(y_test, y_pred_rf, average='weighted'),
        f1_score(y_test, y_pred_cat, average='weighted')
    ]
})
results = results.round(4)
results

,Model,Accuracy,Precision,Recall,F1-Score
0,Logistic Regression,0.7138,0.7214,0.7138,0.7168
1,Random Forest,0.7110,0.7092,0.7110,0.7083
2,CatBoost,0.7176,0.7169,0.7176,0.7168


3.2. ROC-AUC

In [11]:
from sklearn.preprocessing import label_binarize

classes = ['High', 'Low', 'Moderate', 'Severe']

# Chuyển y_test thành dạng nhị phân
y_test_bin = label_binarize(y_test, classes=classes)

# Xác suất dự đoán
log_prob = log_model.predict_proba(X_test_log)
rf_prob = rf_model.predict_proba(X_test)
cat_prob = cat_model.predict_proba(X_test)

# ROC-AUC Logistic Regression
auc_log = roc_auc_score(
    y_test_bin,
    log_prob,
    multi_class='ovr',
    average='weighted'
)

# ROC-AUC Random Forest
auc_rf = roc_auc_score(
    y_test_bin,
    rf_prob,
    multi_class='ovr',
    average='weighted'
)

# ROC-AUC CatBoost
auc_cat = roc_auc_score(
    y_test_bin,
    cat_prob,
    multi_class='ovr',
    average='weighted'
)

print("ROC-AUC Logistic Regression:", round(auc_log, 4))
print("ROC-AUC Random Forest:", round(auc_rf, 4))
print("ROC-AUC CatBoost:", round(auc_cat, 4))

results['ROC-AUC'] = [
    auc_log,
    auc_rf,
    auc_cat
]

results = results.round(4)
results

ROC-AUC Logistic Regression: 0.9179
ROC-AUC Random Forest: 0.9156
ROC-AUC CatBoost: 0.919


,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Logistic Regression,0.7138,0.7214,0.7138,0.7168,0.9179
1,Random Forest,0.7110,0.7092,0.7110,0.7083,0.9156
2,CatBoost,0.7176,0.7169,0.7176,0.7168,0.9190


3.5. Kiểm tra Overfitting

In [12]:
# Logistic Regression
log_train_acc = accuracy_score(y_train, log_model.predict(X_train_log))
log_test_acc = accuracy_score(y_test, y_pred_log)

# Random Forest
rf_train_acc = accuracy_score(y_train, rf_model.predict(X_train))
rf_test_acc = accuracy_score(y_test, y_pred_rf)

# CatBoost
cat_train_acc = accuracy_score(y_train, cat_model.predict(X_train).ravel())
cat_test_acc = accuracy_score(y_test, y_pred_cat)

# Tạo bảng
overfit_check = pd.DataFrame({
    'Model': [
        'Logistic Regression',
        'Random Forest',
        'CatBoost'
    ],
    'Train Accuracy': [
        log_train_acc,
        rf_train_acc,
        cat_train_acc
    ],
    'Test Accuracy': [
        log_test_acc,
        rf_test_acc,
        cat_test_acc
    ]
})

overfit_check['Gap'] = (overfit_check['Train Accuracy'] - overfit_check['Test Accuracy'])

overfit_check.round(4)

,Model,Train Accuracy,Test Accuracy,Gap
0,Logistic Regression,0.7099,0.7138,-0.0040
1,Random Forest,0.7412,0.7110,0.0303
2,CatBoost,0.7192,0.7176,0.0016


In [13]:
# Lấy 10 mẫu từ tập Test
sample = X_test.head(10)

# Dự đoán mức độ Burnout
predicted = cat_model.predict(sample).ravel()

# Xác suất dự đoán
probabilities = cat_model.predict_proba(sample)

# Tạo bảng kết quả
prediction_result = pd.DataFrame({
    'Thực tế': y_test.head(10).values,
    'Dự đoán': predicted,
    'Độ tin cậy (%)': probabilities.max(axis=1) * 100
})

prediction_result['Độ tin cậy (%)'] = prediction_result['Độ tin cậy (%)'].round(2)

display(prediction_result)

,Thực tế,Dự đoán,Độ tin cậy (%)
0,Low,Low,67.82
1,Low,Moderate,71.07
2,High,Moderate,59.75
3,High,High,59.56
4,Low,Low,94.64
5,High,Severe,94.58
6,Low,Low,98.50
7,Low,Low,97.10
8,High,Moderate,46.82
9,High,High,53.19


4. Mô hình dự đoán

In [14]:
new_person = pd.DataFrame([{
    "age": 20,
    "gender": "Female",
    "survey_year": 2026,
    "country": 'Vietnam',
    "job_role": "Data Analyst",
    "seniority_level": "Junior",
    "years_experience": 2,
    "work_mode": "Hybrid",
    "salary_usd": 50000,
    "work_hours_per_week": 15,
    "sleep_hours_per_night": 6.5,
    "uses_therapy": 0,
    "ai_tools_daily": 1,
    "deadline_pressure_score": 5,
    "stress_score": 5,
    "phq9_score": 8,
    "gad7_score": 9,
    "seeks_mental_health_support": 0,
    "job_change_intention": 1,
    "work_life_balance_ratio": 2 
}])
display(new_person)

,age,gender,survey_year,country,job_role,seniority_level,years_experience,work_mode,salary_usd,work_hours_per_week,sleep_hours_per_night,uses_therapy,ai_tools_daily,deadline_pressure_score,stress_score,phq9_score,gad7_score,seeks_mental_health_support,job_change_intention,work_life_balance_ratio
0,20,Female,2026,Vietnam,Data Analyst,Junior,2,Hybrid,50000,15,6.5,0,1,5,5,8,9,0,1,2


In [15]:
# 1. Load bộ tiền xử lý có sẵn
preprocessor = joblib.load("burnout_preprocessor.joblib")
# 2. Chuẩn bị dữ liệu người mới
new_input = new_person.copy()

new_input["burnout_score"] = np.nan
new_input["total_score"] = np.nan
new_input["phq9_category"] = None
new_input["gad7_category"] = None
new_input["hourly_wage_usd"] = np.nan

# 3. Tiền xử lý
new_processed = pd.DataFrame(
    preprocessor.transform(new_input),
    columns=preprocessor.get_feature_names_out(),
    index=new_input.index
)

# 4. Chọn đúng các feature mô hình cần
X_new = new_processed[X_train_log.columns].copy()

# 5. Chạy mô hình
prediction = log_model.predict(X_new)

# 6. Xuất kết quả
print("Số feature:", X_new.shape[1])
print("Mức độ Kiệt sức dự đoán:", prediction[0])

Số feature: 11
Mức độ Kiệt sức dự đoán: Low


5. Hàm tính toán của model

In [16]:

import pandas as pd
import numpy as np

# Lấy đúng thứ tự feature và lớp từ model đã huấn luyện
features = X_train_log.columns.tolist()
classes = log_model.classes_

coef_df = pd.DataFrame(
    log_model.coef_,
    index=classes,
    columns=features
)

coef_df.insert(0, "Intercept (beta_0)", log_model.intercept_)

print("Các lớp:", classes)
print("Số feature:", len(features))
print("\nHệ số beta của Logistic Regression:")
display(coef_df.round(6))

print("\nCÔNG THỨC TUYẾN TÍNH CỦA CÁC LỚP:")

for k, class_name in enumerate(classes):
    terms = [f"{log_model.intercept_[k]:.6f}"]

    for feature, beta in zip(features, log_model.coef_[k]):
        terms.append(f"({beta:+.6f}) * {feature}")

    print(f"\nz_{class_name} =")
    print(" ".join(terms))

# coef_df.to_csv("logistic_regression_coefficients.csv")

print("CÔNG THỨC XÁC SUẤT SOFTMAX:\n")

for class_name in classes:
    print(f"P(Y = {class_name} | X) =")
    print(
        f"exp(z_{class_name}) / "
        f"(" + " + ".join(
            [f"exp(z_{c})" for c in classes]
        ) + ")"
    )
    print()


Các lớp: ['High' 'Low' 'Moderate' 'Severe']
Số feature: 11

Hệ số beta của Logistic Regression:


,Intercept (beta_0),numeric__years_experience,numeric__work_hours_per_week,numeric__sleep_hours_per_night,numeric__work_life_balance_ratio,numeric__uses_therapy,numeric__deadline_pressure_score,numeric__stress_score,numeric__phq9_score,numeric__gad7_score,numeric__seeks_mental_health_support,numeric__job_change_intention
High,-0.494734,-0.002925,0.084171,-0.542403,-1.287781,0.141800,-0.137329,0.417867,0.086347,0.116700,0.124119,0.123109
Low,4.397840,0.005772,-0.288883,1.943318,4.459271,-0.398046,0.417743,-1.423855,-0.282532,-0.360805,-0.392397,-0.354421
Moderate,2.113386,0.001910,-0.096654,0.723114,1.608731,-0.134167,0.123803,-0.417819,-0.067492,-0.087308,-0.130450,-0.089002
Severe,-6.016492,-0.004757,0.301365,-2.124029,-4.780221,0.390413,-0.404216,1.423808,0.263677,0.331412,0.398728,0.320313



CÔNG THỨC TUYẾN TÍNH CỦA CÁC LỚP:

z_High =
-0.494734 (-0.002925) * numeric__years_experience (+0.084171) * numeric__work_hours_per_week (-0.542403) * numeric__sleep_hours_per_night (-1.287781) * numeric__work_life_balance_ratio (+0.141800) * numeric__uses_therapy (-0.137329) * numeric__deadline_pressure_score (+0.417867) * numeric__stress_score (+0.086347) * numeric__phq9_score (+0.116700) * numeric__gad7_score (+0.124119) * numeric__seeks_mental_health_support (+0.123109) * numeric__job_change_intention

z_Low =
4.397840 (+0.005772) * numeric__years_experience (-0.288883) * numeric__work_hours_per_week (+1.943318) * numeric__sleep_hours_per_night (+4.459271) * numeric__work_life_balance_ratio (-0.398046) * numeric__uses_therapy (+0.417743) * numeric__deadline_pressure_score (-1.423855) * numeric__stress_score (-0.282532) * numeric__phq9_score (-0.360805) * numeric__gad7_score (-0.392397) * numeric__seeks_mental_health_support (-0.354421) * numeric__job_change_intention

z_Moderate =

In [17]:
# Kiểm tra
x = X_test_log.iloc[[0]]

z = x.to_numpy() @ log_model.coef_.T + log_model.intercept_

# Tính xác suất bằng công thức Softmax
exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
probabilities = exp_z / exp_z.sum(axis=1, keepdims=True)

result = pd.DataFrame({
    "Lớp": log_model.classes_,
    "Điểm tuyến tính z": z[0],
    "exp(z) đã chuẩn hóa": probabilities[0]
})

print("KẾT QUẢ TÍNH XÁC SUẤT CHO MỘT DÒNG DỮ LIỆU:")
display(result.round(6))

print("Tổng xác suất:", round(probabilities.sum(), 6))

prob_model = log_model.predict_proba(x)

print(
    "Khớp với xác suất của mô hình:",
    np.allclose(probabilities, prob_model)
)


KẾT QUẢ TÍNH XÁC SUẤT CHO MỘT DÒNG DỮ LIỆU:


,Lớp,Điểm tuyến tính z,exp(z) đã chuẩn hóa
0,High,-0.003561,0.025541
1,Low,3.145422,0.595418
2,Moderate,2.693605,0.378966
3,Severe,-5.835466,0.000075


Tổng xác suất: 1.0
Khớp với xác suất của mô hình: True
